**I have Performed PyTorch implementation of a self-pruning convolutional neural network for CIFAR-10, using learnable sigmoid gates, STE-based hard masking, and L1 sparsity regularization to jointly optimize accuracy and model sparsity during training.**

In [3]:
"""
Self-Pruning CNN for CIFAR-10 — FIXED VERSION
==============================================
Fixes applied:
- Gate init mean=0.0 (starts at decision boundary, L1 pushes toward 0)
- LR_GATE = 1e-2 (faster gate death)
- sparsity_loss uses .abs().mean() (stronger L1 push)
- STE hard binary mask, separate optimizers, label smoothing, grad clipping
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os, time, json

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE   = 128
EPOCHS       = 30
LR_MAIN      = 1e-3
LR_GATE      = 1e-2        # ✅ FIX 2: was 5e-3, doubled for faster gate death
WEIGHT_DECAY = 1e-4
LAMBDAS      = [1e-3, 1e-2, 5e-2]
PRUNE_THRESH = 0.5
SAVE_DIR     = "output_cnn"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device : {DEVICE}")
print(f"Epochs : {EPOCHS}  |  Lambdas : {LAMBDAS}")


# ═══════════════════════════════════════════════
# 1. STE — Straight-Through Estimator
# ═══════════════════════════════════════════════
class STEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, gates, threshold=0.5):
        return (gates >= threshold).float()

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None


def ste_threshold(gates, threshold=0.5):
    return STEFunction.apply(gates, threshold)


# ═══════════════════════════════════════════════
# 2. PrunableLinear
# ═══════════════════════════════════════════════
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight      = nn.Parameter(torch.empty(out_features, in_features))
        self.bias_param  = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))
        self._init_parameters()

    def _init_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=0.01)
        # ✅ FIX 1: mean=0.0 → sigmoid(0)=0.5 → right at boundary
        # L1 immediately pushes gate_scores negative → hard mask becomes 0
        nn.init.normal_(self.gate_scores, mean=0.0, std=0.01)

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)
        mask  = ste_threshold(gates, 0.5) if self.training else (gates >= 0.5).float()
        return F.linear(x, self.weight * mask, self.bias_param)

    def get_soft_gates(self):
        return torch.sigmoid(self.gate_scores)

    def get_hard_mask(self):
        with torch.no_grad():
            return (torch.sigmoid(self.gate_scores) >= 0.5).float()

    def sparsity(self):
        mask = self.get_hard_mask()
        return (mask == 0).float().mean().item()


# ═══════════════════════════════════════════════
# 3. SelfPruningCNN
# ═══════════════════════════════════════════════
class SelfPruningCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 — 64 filters dual conv (friend's design)
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.15),
        
            # Block 2 — 128 filters dual conv
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.20),
        
            # Block 3 — 256 filters dual conv
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )


        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            PrunableLinear(256 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            PrunableLinear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            PrunableLinear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def prunable_layers(self):
        return [m for m in self.modules() if isinstance(m, PrunableLinear)]

    def sparsity_loss(self):
        all_gates = torch.cat([
            l.get_soft_gates().view(-1) for l in self.prunable_layers()
        ])
        # ✅ FIX 3: .abs().mean() gives stronger individual gate push toward 0
        return all_gates.abs().mean()

    def overall_sparsity(self):
        total, pruned = 0, 0
        for layer in self.prunable_layers():
            mask    = layer.get_hard_mask()
            pruned += (mask == 0).sum().item()
            total  += mask.numel()
        return 100.0 * pruned / total


# ═══════════════════════════════════════════════
# 4. Data Loaders
# ═══════════════════════════════════════════════
def get_loaders():
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_ds = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True),
        DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True),
    )


# ═══════════════════════════════════════════════
# 5. Loss, Optimizer, Training
# ═══════════════════════════════════════════════
def total_loss(logits, targets, model, lam, smoothing=0.1):
    log_probs   = F.log_softmax(logits, dim=1)
    smooth_loss = -log_probs.mean(dim=1).mean()
    true_loss   = F.nll_loss(log_probs, targets)
    cls_loss    = (1 - smoothing) * true_loss + smoothing * smooth_loss
    return cls_loss + lam * model.sparsity_loss()


def make_optimizers(model):
    gate_p, other_p = [], []
    for name, p in model.named_parameters():
        (gate_p if "gate_scores" in name else other_p).append(p)
    optimizer = optim.AdamW([
        {"params": other_p, "lr": LR_MAIN, "weight_decay": WEIGHT_DECAY},
        {"params": gate_p,  "lr": LR_GATE, "weight_decay": 0.0},
    ])
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    return optimizer, scheduler


def train_epoch(model, loader, optimizer, lam):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = total_loss(logits, labels, model, lam)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct      += logits.argmax(1).eq(labels).sum().item()
        total        += imgs.size(0)
    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        correct += model(imgs).argmax(1).eq(labels).sum().item()
        total   += imgs.size(0)
    return 100.0 * correct / total


# ═══════════════════════════════════════════════
# 6. Experiment Runner
# ═══════════════════════════════════════════════
def run_experiment(lam, train_loader, test_loader):
    print(f"\n{'='*60}")
    print(f"  Lambda = {lam}")
    print(f"{'='*60}")

    model                = SelfPruningCNN().to(DEVICE)
    optimizer, scheduler = make_optimizers(model)
    history = {"train_acc": [], "test_acc": [], "sparsity": [], "loss": []}

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, lam)
        te_acc          = evaluate(model, test_loader)
        sparsity        = model.overall_sparsity()
        scheduler.step()

        history["train_acc"].append(tr_acc)
        history["test_acc"].append(te_acc)
        history["sparsity"].append(sparsity)
        history["loss"].append(tr_loss)

        if epoch == 1 or epoch % 5 == 0:
            print(f"  Epoch {epoch:>2d}/{EPOCHS} | Loss {tr_loss:.4f} | "
                  f"Train {tr_acc:.1f}% | Test {te_acc:.1f}% | "
                  f"Sparsity {sparsity:.1f}% | {time.time()-t0:.1f}s")

    final_acc      = evaluate(model, test_loader)
    final_sparsity = model.overall_sparsity()
    print(f"\n  Final → Test Accuracy: {final_acc:.2f}%  |  Sparsity: {final_sparsity:.2f}%")

    all_gates = torch.cat([
        l.get_soft_gates().detach().view(-1).cpu()
        for l in model.prunable_layers()
    ]).numpy()

    return {
        "lambda": lam, "test_accuracy": round(final_acc, 2),
        "sparsity": round(final_sparsity, 2),
        "history": history, "gates": all_gates,
    }


# ═══════════════════════════════════════════════
# 7. Plots
# ═══════════════════════════════════════════════
def save_plots(results):
    colors = ["#2196F3", "#FF9800", "#E91E63"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, res in enumerate(results):
        ep = range(1, EPOCHS + 1)
        axes[0].plot(ep, res["history"]["test_acc"], color=colors[i], lw=2, label=f"λ={res['lambda']}")
        axes[1].plot(ep, res["history"]["sparsity"], color=colors[i], lw=2, label=f"λ={res['lambda']}")
        axes[2].plot(ep, res["history"]["loss"],     color=colors[i], lw=2, label=f"λ={res['lambda']}")
    for ax, title, ylabel in zip(axes,
        ["Test Accuracy over Epochs", "Sparsity (%) over Epochs", "Training Loss over Epochs"],
        ["Test Accuracy (%)", "Sparsity (%)", "Loss"]):
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle("Self-Pruning CNN — CIFAR-10 (STE Gates, Fixed)", fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=150, bbox_inches="tight")
    plt.close()

    fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5))
    for ax, res in zip(axes, results):
        ax.hist(res["gates"], bins=80, color="#1976D2", edgecolor="none", alpha=0.85)
        ax.axvline(0.5, color="red", linestyle="--", lw=1.5, label="threshold=0.5")
        ax.set_title(f"λ={res['lambda']}\nAcc={res['test_accuracy']}%  Sparsity={res['sparsity']}%",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Soft Gate Value"); ax.set_ylabel("Count")
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle("Gate Distributions — SelfPruningCNN (Fixed)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/gate_distributions.png", dpi=150, bbox_inches="tight")
    plt.close()

    lam_labels = [str(r["lambda"]) for r in results]
    accs  = [r["test_accuracy"] for r in results]
    spars = [r["sparsity"]      for r in results]
    x, width = np.arange(len(lam_labels)), 0.35
    fig, ax1 = plt.subplots(figsize=(9, 6))
    ax2 = ax1.twinx()
    bars1 = ax1.bar(x - width/2, accs,  width, label="Test Accuracy (%)", color="#1976D2", alpha=0.85)
    bars2 = ax2.bar(x + width/2, spars, width, label="Sparsity (%)",       color="#E91E63", alpha=0.85)
    ax1.set_xlabel("Lambda (λ)", fontsize=12)
    ax1.set_ylabel("Test Accuracy (%)", color="#1976D2", fontsize=12)
    ax2.set_ylabel("Sparsity (%)",      color="#E91E63", fontsize=12)
    ax1.set_xticks(x); ax1.set_xticklabels(lam_labels)
    ax1.set_ylim(0, 105); ax2.set_ylim(0, 105)
    for bar in bars1:
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=10)
    for bar in bars2:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=10)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, loc="upper left")
    ax1.set_title("Accuracy vs Sparsity Trade-off — SelfPruningCNN", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/lambda_tradeoff.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plots saved to ./{SAVE_DIR}/")


# ═══════════════════════════════════════════════
# 8. Main
# ═══════════════════════════════════════════════
if __name__ == "__main__":
    train_loader, test_loader = get_loaders()
    results = []

    for lam in LAMBDAS:
        res = run_experiment(lam, train_loader, test_loader)
        results.append(res)

    summary = [{"lambda": r["lambda"],
                "test_accuracy_%": r["test_accuracy"],
                "sparsity_%": r["sparsity"]} for r in results]
    with open(f"{SAVE_DIR}/results_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\n\n" + "="*60)
    print("  FINAL RESULTS — SelfPruningCNN (Fixed)")
    print("="*60)
    print(f"  {'Lambda':<12} {'Test Accuracy':>14}   {'Sparsity':>10}")
    print(f"  {'─'*42}")
    for s in summary:
        print(f"  {s['lambda']:<12} {s['test_accuracy_%']:>13.2f}%   {s['sparsity_%']:>9.2f}%")
    print("="*60)

    save_plots(results)
    print("\nAll done!")

Device : cuda
Epochs : 30  |  Lambdas : [0.001, 0.01, 0.05]

  Lambda = 0.001
  Epoch  1/30 | Loss 1.8182 | Train 38.7% | Test 56.5% | Sparsity 61.6% | 21.7s
  Epoch  5/30 | Loss 1.2021 | Train 68.8% | Test 75.9% | Sparsity 70.0% | 22.3s
  Epoch 10/30 | Loss 1.0248 | Train 77.3% | Test 83.2% | Sparsity 71.2% | 22.6s
  Epoch 15/30 | Loss 0.9342 | Train 81.4% | Test 85.7% | Sparsity 70.9% | 22.5s
  Epoch 20/30 | Loss 0.8692 | Train 84.4% | Test 88.1% | Sparsity 70.3% | 22.9s
  Epoch 25/30 | Loss 0.8338 | Train 86.0% | Test 89.1% | Sparsity 70.0% | 22.2s
  Epoch 30/30 | Loss 0.8166 | Train 86.8% | Test 89.2% | Sparsity 69.9% | 21.9s

  Final → Test Accuracy: 89.22%  |  Sparsity: 69.91%

  Lambda = 0.01
  Epoch  1/30 | Loss 1.8227 | Train 38.5% | Test 56.5% | Sparsity 62.9% | 22.2s
  Epoch  5/30 | Loss 1.1990 | Train 69.2% | Test 78.4% | Sparsity 72.7% | 22.4s
  Epoch 10/30 | Loss 1.0237 | Train 77.5% | Test 83.4% | Sparsity 75.3% | 21.9s
  Epoch 15/30 | Loss 0.9319 | Train 81.8% | Test 86